In [2]:
import os
os.environ["OPENAI_API_KEY"] = "None"
os.environ["SGLANG_API_URL"] = "http://localhost:10086/v1"

In [7]:
import requests
url = "http://localhost:10086/v1/chat/completions"

data = {
    "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "messages": [
        {"role": "user", "content": "List 3 countries and their capitals."}
    ],
    "return_logprob": True
}

response = requests.post(url, json=data)
print(response.json())

{'id': 'e6a694d9dc034ec3b90456ee51761e9c', 'object': 'chat.completion', 'created': 1736449642, 'model': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Here are 3 countries and their capitals:\n\n1. Country: Japan\n   Capital: Tokyo\n\n2. Country: Australia\n   Capital: Canberra\n\n3. Country: Brazil\n   Capital: Brasília'}, 'logprobs': None, 'finish_reason': 'stop', 'matched_stop': 128009}], 'usage': {'prompt_tokens': 43, 'total_tokens': 86, 'completion_tokens': 43, 'prompt_tokens_details': None}}


In [3]:
import requests
response = requests.post(
    url = "http://localhost:10086/generate",
    json={
        "text": "The capital of France is",
        "sampling_params": {
            "temperature": 0,
            "max_new_tokens": 32,
        },
        "return_logprob": True,
        # "logprob_start_len": 0,
    },
)
print(response.json())


{'text': ' a city of many faces. It is a city of history, culture, and art. It is a city of fashion, food, and wine. It is', 'meta_info': {'prompt_tokens': 6, 'completion_tokens': 32, 'completion_tokens_wo_jump_forward': 32, 'cached_tokens': 3, 'finish_reason': {'type': 'length', 'length': 32}, 'input_token_logprobs': [], 'output_token_logprobs': [[-1.742870569229126, 264, None], [-1.5830035209655762, 3363, None], [-1.1122164726257324, 315, None], [-2.7300384044647217, 1690, None], [-1.3603432178497314, 12580, None], [-0.6628015041351318, 13, None], [-1.8872967958450317, 1102, None], [-0.9070780277252197, 374, None], [-1.1356332302093506, 264, None], [-1.0052813291549683, 3363, None], [-0.6354659199714661, 315, None], [-1.9960246086120605, 3925, None], [-0.7298474311828613, 11, None], [-1.7198235988616943, 7829, None], [-0.2846848964691162, 11, None], [-0.9945531487464905, 323, None], [-1.4361846446990967, 1989, None], [-0.7132233381271362, 13, None], [-0.4154910445213318, 1102, None],

'http://localhost:10086/generate'

In [16]:
response.json()['meta_info']['normalized_prompt_logprob']#['normalized_prompt_logprob']#text

-13.149120330810547

In [26]:
from reasoners.benchmark import BWEvaluator
import json

with open('examples/CoT/blocksworld/prompts/pool_prompt_v1.json') as f:
    prompt = json.load(f)
evaluator = BWEvaluator(config_file='examples/CoT/blocksworld/data/bw_config.yaml',
                        domain_file='examples/CoT/blocksworld/data/generated_domain.pddl',
                        data_path='examples/CoT/blocksworld/data/split_v1/split_v1_step_4_data.json',
                        init_prompt=prompt)
prompt = evaluator.sample_prompt(shuffle_prompt=False, num_shot=4)
example = evaluator.full_dataset[1]
cot_inputs = (prompt['icl'].replace('<init_state>', example["init"])
                           .replace('<goals>', example["goal"])
                           .replace('<action>', ''))

In [27]:
output = model.generate(cot_inputs,
                        hide_input=True,
                        eos_token_id='\n[').text[0][:-1].strip()

An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given, sleeping for 1 seconds
An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given, sleeping for 2 seconds
An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given, sleeping for 3 seconds
An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given, sleeping for 4 seconds
An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given, sleeping for 5 seconds
An Error Occured: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be give

KeyboardInterrupt: 

In [ ]:
print(output)

Based on the provided information, I will analyze each plan to check if it is valid.

**Plan 1:**
- Initial conditions: red block is clear, orange block is clear, hand is empty, orange block is on top of blue block, red block is on table, blue block is on table.
- Goal: blue block is on top of orange block.
- Plan: unstack orange block from on top of blue block, put down orange block, pick up blue block, stack blue block on top of orange block.
- This plan is valid.

**Plan 2:**
- Initial conditions: blue block is clear, orange block is clear, hand is empty, red block is on top of yellow block, orange block is on top of red block, blue block is on table, yellow block is on table.
- Goal: blue block is on top of yellow block and orange block is on top of blue block.
- Plan: unstack orange block from on top of red block, put down orange block, unstack red block from on top of yellow block, put down red block, pick up blue block, stack blue block on top of yellow block, pick up orange blo

In [4]:
from reasoners import WorldModel, LanguageModel, SearchConfig, State, Reasoner
from reasoners.algorithm import BeamSearch, MCTS
import reasoners.benchmark.bw_utils as utils
from typing import NamedTuple
import copy
import numpy as np

In [5]:
BWAction = str

class BWStateRAP(NamedTuple):
    step_idx: int
    last_blocks_state: str
    blocks_state: str
    buffered_action: BWAction


class BlocksWorldModelRAP(WorldModel):
    def __init__(self,
                 base_model: LanguageModel,
                 prompt: dict,
                 max_steps: int = 4,
                 batch_size: int = 1) -> None:
        super().__init__()
        self.max_steps = max_steps
        self.base_model = base_model
        self.prompt = prompt
        self.batch_size = batch_size

    def init_state(self) -> BWStateRAP:
        return BWStateRAP(step_idx=0, last_blocks_state="", blocks_state=utils.
                       extract_init_state(self.example), buffered_action="")

    def step(self, state: BWStateRAP, action: BWAction) -> tuple[BWStateRAP, dict]:
        state = copy.deepcopy(state)
        blocks_state = state.blocks_state
        step_idx = state.step_idx
        blocks_state = self.update_blocks(blocks_state, action)
        new_buffered_action = action if state.buffered_action == "" else ""

        state = BWStateRAP(step_idx=step_idx + 1,
                        last_blocks_state=state.blocks_state,
                        blocks_state=blocks_state,
                        buffered_action=new_buffered_action)
        return state, {"goal_reached": utils.goal_check(utils.extract_goals(self.example), blocks_state)}

    def update_blocks(self, block_states: str, action: BWAction) -> str:
        if "pick" in action:
            key = "world_update_pickup"
        elif "unstack" in action:
            key = "world_update_unstack"
        elif "put" in action:
            key = "world_update_putdown"
        elif "stack" in action:
            key = "world_update_stack"
        else:
            raise ValueError("Invalid action")
        world_update_prompt = self.prompt[key].format(block_states, action.capitalize() + ".")
        world_output = self.base_model.generate([world_update_prompt],
                                                # eos_token_id="\n",
                                                # hide_input=True,
                                                temperature=0).text[0].strip()
        new_state = utils.apply_change(world_output, block_states)
        return new_state

    def is_terminal(self, state: BWStateRAP) -> bool:
        if utils.goal_check(utils.extract_goals(self.example), state.blocks_state)[0]:
            return True
        elif state.step_idx == self.max_steps:
            return True
        return False

In [12]:
class BWConfigRAP(SearchConfig):
    def __init__(self,
                 base_model: LanguageModel,
                 prompt: dict,
                 batch_size: int = 1,
                 reward_alpha: float = 0.5,
                 goal_reward_default: float = 0.,
                 goal_reached_reward: float = 100.) -> None:
        super().__init__()
        self.base_model = base_model
        self.example = None
        self.prompt = prompt
        self.batch_size = batch_size
        self.reward_alpha = reward_alpha
        self.goal_reward_default = goal_reward_default
        self.goal_reached_reward = goal_reached_reward

    def get_actions(self, state: BWStateRAP) -> list[BWAction]:
        blocks_state = state.blocks_state
        return utils.generate_all_actions(blocks_state)

    def fast_reward(self, state: BWStateRAP, action: BWAction) -> tuple[float, dict]:
        if state.buffered_action == "":
            current_blocks_state = state.blocks_state
        else:
            current_blocks_state = state.last_blocks_state
        previous_action = state.buffered_action + "\n" if state.buffered_action != "" else ""
        
        # every two steps, we will also reduce the icl examples by 2 steps
        # so that the distribution of step length in examples is more reasonable
        icl_template = self.prompt["icl_list"][state.step_idx // 2]
        
        inputs = (icl_template.replace("<init_state>", current_blocks_state)
                              .replace("<goals>", utils.extract_goals(self.example, return_raw=True))
                              .replace("<action>", previous_action))
        intuition = self.base_model.get_loglikelihood(inputs, [inputs + action])[0]

        self_eval_prompt = (self.prompt["self-eval"]
                                .replace("<init_state>", current_blocks_state)
                                .replace("<goals>", utils.extract_goals(self.example, return_raw=True))
                                .replace("<action>", action))
        self_eval = self.base_model.get_loglikelihood(self_eval_prompt, [self_eval_prompt + "good"])[0]

        return (self.calculate_reward(intuition, self_eval),
                {'intuition': intuition, "self_eval": self_eval})

    def calculate_reward(self, intuition, self_eval, goal_reached=None) -> float:
        # to provide a unified interface for reward and fast_reward
        if goal_reached is None:
            goal_reward = self.goal_reward_default
        elif goal_reached[0]:
            goal_reward = self.goal_reached_reward
        else:
            goal_reward = goal_reached[1]
        return (intuition + self_eval) * self.reward_alpha + goal_reward * (1 - self.reward_alpha)

    def reward(self, state: BWStateRAP, action: BWAction,
               intuition: float = None,
               self_eval: float = None,
               goal_reached: tuple[bool, float] = None) -> tuple[float, dict]:
        return (self.calculate_reward(intuition, self_eval, goal_reached),
                {'intuition': intuition, 'goal_reached': goal_reached})

In [13]:
from reasoners.benchmark import BWEvaluator
import json

with open('examples/CoT/blocksworld/prompts/pool_prompt_v1.json') as f:
    prompt = json.load(f)
evaluator = BWEvaluator(config_file='examples/CoT/blocksworld/data/bw_config.yaml',
                        domain_file='examples/CoT/blocksworld/data/generated_domain.pddl',
                        data_path='examples/CoT/blocksworld/data/split_v1/split_v1_step_4_data.json',
                        init_prompt=prompt)
prompt = evaluator.sample_prompt(shuffle_prompt=False, num_shot=4)
example = evaluator.full_dataset[1]
cot_inputs = (prompt['icl'].replace('<init_state>', example["init"])
                           .replace('<goals>', example["goal"])
                           .replace('<action>', ''))

In [1]:
from reasoners.lm.openai_model import OpenAIModel
import os

os.environ["OPENAI_API_KEY"] = "None"
os.environ["SGLANG_API_URL"] = "http://localhost:10086/v1"

model = OpenAIModel(
    model="meta-llama/Llama-3.1-8B",
    backend="sglang",
    is_instruct_model=False,
)

/data/irving/anaconda3/envs/sglang/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/irving/anaconda3/envs/sglang/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [15]:
world_model = BlocksWorldModelRAP(base_model=model, prompt=prompt, max_steps=4)
config = BWConfigRAP(base_model=model, prompt=prompt)
algorithm = MCTS(depth_limit=4, disable_tqdm=False, output_trace_in_each_iter=True, n_iters=10)
reasoner_rap = Reasoner(world_model=world_model, search_config=config, search_algo=algorithm)
result_rap = reasoner_rap(example)
print(result_rap)

Error: no successful change
the blue block is no longer clear.
[state 1] i have that
['the blue block is clear', 'the orange block is clear', 'the hand is holding the blue block', 'the orange block is on top of the red block', 'the red block is on the table', 'the blue block is in the hand']


Exception: ERROR

In [7]:
from reasoners.lm.openai_model import OpenAIModel

model = OpenAIModel(
    model="meta-llama/Llama-3.1-8B",
    backend="sglang",
    is_instruct_model=False,
)

result = model.generate(
    prompt=['pick up the red block'],
    temperature=0,
    max_tokens = 32,
    logprobs=True
)

print(result)


KeyboardInterrupt: 

In [4]:
model.get_loglikelihood("The Cat", ["The Cat sits on the couch","The Cat sits on the sofa"])

array([-7.14631128, -7.23381138])

In [6]:
result.text[0].strip()

'and place it on the blue block\npick up the red block and place it on the blue block\npick up the red block and place it on the blue'

In [17]:
# Import required libraries
import sglang as sgl
from sglang.api import set_default_backend
from sglang import RuntimeEndpoint, function, gen

# Set the default backend
set_default_backend(RuntimeEndpoint("http://localhost:10086"))

# Define the tool_use function
@sgl.function
def helper(s, inputs, action):
    s += "Input" + sgl.gen("response", choices=["pick up the red block", "put down the blue block"])

# Run one case
state = helper.run(input)
meta_info = state.get_meta_info("response")


questions: What is 5 + 5?
choice: calculator


In [23]:
meta_info['normalized_prompt_logprobs']#.normalized_prompt_logprobs

[-1.8299658298492432, -15.552621841430664, -7.206484794616699]